# Gold: Weather Monthly Summary (v1) — SQL

SQL-cell equivalent of `gold_weather_monthly_summary_v1.ipynb`, using
`%%sql` magic instead of the DataFrame API. Same result: reads the
`noaa_weather.daily_observations` bronze table, nulls out NOAA's
missing-value sentinels inline, and persists
`noaa_weather.gold_weather_monthly_summary_v1`.

Requires the bronze table to already exist — run
`data-generators/bronze_ingest_noaa_weather.ipynb` first.

`%%sql` here follows Databricks-style notebook cell magic. If running
outside a platform that provides it natively, install and load
`sparksql-magic` first: `pip install sparksql-magic` then
`%load_ext sparksql_magic` (bound to the `spark` session created below).

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Transform + persist: monthly weather summary

`CASE WHEN` inline nulls out the `9999.9`/`999.9`/`99.99` sentinels before
aggregating. `CREATE OR REPLACE TABLE` keeps this re-runnable.

In [ ]:
%%sql
CREATE OR REPLACE TABLE noaa_weather.gold_weather_monthly_summary_v1 AS
SELECT
    date_format(CAST(DATE AS DATE), 'yyyy-MM') AS month,
    ROUND(AVG(CASE WHEN TEMP = 9999.9 THEN NULL ELSE TEMP END), 1) AS avg_temp_f,
    ROUND(MAX(CASE WHEN `MAX` = 9999.9 THEN NULL ELSE `MAX` END), 1) AS max_temp_f,
    ROUND(MIN(CASE WHEN `MIN` = 9999.9 THEN NULL ELSE `MIN` END), 1) AS min_temp_f,
    ROUND(SUM(CASE WHEN PRCP = 99.99 THEN NULL ELSE PRCP END), 2) AS total_precip_in,
    ROUND(AVG(CASE WHEN WDSP = 999.9 THEN NULL ELSE WDSP END), 1) AS avg_wind_speed_kt
FROM noaa_weather.daily_observations
GROUP BY date_format(CAST(DATE AS DATE), 'yyyy-MM')
ORDER BY month

## Verify

In [ ]:
%%sql
SELECT COUNT(*) AS total_months FROM noaa_weather.gold_weather_monthly_summary_v1

In [ ]:
%%sql
SELECT * FROM noaa_weather.gold_weather_monthly_summary_v1
ORDER BY month
LIMIT 12